Регрессия: предсказание SI (Selectivity Index)
Важно: SI = CC50 / IC50 — при использовании IC50/CC50 как признаков 
будет утечка данных. Используем только молекулярные дескрипторы.

In [1]:
import os
os.makedirs("plots", exist_ok=True)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.feature_selection import VarianceThreshold
import warnings

warnings.filterwarnings("ignore")

## Данные

In [2]:
df = pd.read_excel("data/data.xlsx", index_col=0)
targets = ["IC50, mM", "CC50, mM", "SI"]
features = [c for c in df.columns if c not in targets]

X = df[features]
# log1p(SI) — нормализуем сильно правостороннее распределение
y = np.log1p(df["SI"])

selector = VarianceThreshold(threshold=0.0)
X = X.fillna(X.median())
X_sel = selector.fit_transform(X)
sel_features = [features[i] for i in selector.get_support(indices=True)]
X = pd.DataFrame(X_sel, columns=sel_features)
print(f"Признаков после фильтрации: {X.shape[1]}")
print("SI: описание исходных значений:")
print(df["SI"].describe().to_string())

cv = KFold(n_splits=5, shuffle=True, random_state=42)


def cv_rmse(model, X, y, cv):
    return -cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error").mean()


def cv_r2(model, X, y, cv):
    return cross_val_score(model, X, y, cv=cv, scoring="r2").mean()

Признаков после фильтрации: 192
SI: описание исходных значений:
count     1001.000000
mean        72.508823
std        684.482739
min          0.011489
25%          1.433333
50%          3.846154
75%         16.566667
max      15620.600000


## Базовые модели

In [3]:
print("\n── Базовые модели (дефолтные параметры) ──")
base_models = {
    "Ridge":            Pipeline([("sc", StandardScaler()), ("m", Ridge())]),
    "Lasso":            Pipeline([("sc", StandardScaler()), ("m", Lasso(max_iter=5000))]),
    "ElasticNet":       Pipeline([("sc", StandardScaler()), ("m", ElasticNet(max_iter=5000))]),
    "RandomForest":     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
}
for name, model in base_models.items():
    rmse = cv_rmse(model, X, y, cv)
    r2 = cv_r2(model, X, y, cv)
    print(f"  {name:22s}: RMSE={rmse:.4f}  R²={r2:.4f}")


── Базовые модели (дефолтные параметры) ──
  Ridge                 : RMSE=1.3597  R²=0.1032
  Lasso                 : RMSE=1.4522  R²=-0.0031
  ElasticNet            : RMSE=1.4522  R²=-0.0031
  RandomForest          : RMSE=1.1838  R²=0.3308
  GradientBoosting      : RMSE=1.2023  R²=0.3087


## GridSearchCV

In [4]:
print("\n── GridSearchCV: Ridge ──")
gs_ridge = GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("m", Ridge())]),
    {"m__alpha": [0.01, 0.1, 1, 10, 100]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
gs_ridge.fit(X, y)
print(f"  Best: {gs_ridge.best_params_}  RMSE={-gs_ridge.best_score_:.4f}")

print("\n── GridSearchCV: ElasticNet ──")
gs_en = GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("m", ElasticNet(max_iter=10000))]),
    {"m__alpha": [0.001, 0.01, 0.1], "m__l1_ratio": [0.2, 0.5, 0.8]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
gs_en.fit(X, y)
print(f"  Best: {gs_en.best_params_}  RMSE={-gs_en.best_score_:.4f}")

print("\n── GridSearchCV: RandomForest ──")
gs_rf = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {"n_estimators": [100, 200], "max_depth": [None, 10, 20], "min_samples_split": [2, 5]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
gs_rf.fit(X, y)
print(f"  Best: {gs_rf.best_params_}  RMSE={-gs_rf.best_score_:.4f}")

print("\n── GridSearchCV: GradientBoosting ──")
gs_gb = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1, 0.2], "max_depth": [3, 5]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
gs_gb.fit(X, y)
print(f"  Best: {gs_gb.best_params_}  RMSE={-gs_gb.best_score_:.4f}")


── GridSearchCV: Ridge ──
  Best: {'m__alpha': 100}  RMSE=1.3251

── GridSearchCV: ElasticNet ──
  Best: {'m__alpha': 0.1, 'm__l1_ratio': 0.2}  RMSE=1.3176

── GridSearchCV: RandomForest ──
  Best: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}  RMSE=1.1720

── GridSearchCV: GradientBoosting ──
  Best: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100}  RMSE=1.1958


## Сравнение

In [5]:
print("\n── Итоговое сравнение RMSE (tuned) ──")
tuned = {
    "Ridge":            -gs_ridge.best_score_,
    "ElasticNet":       -gs_en.best_score_,
    "RandomForest":     -gs_rf.best_score_,
    "GradientBoosting": -gs_gb.best_score_,
}
for name, rmse in sorted(tuned.items(), key=lambda x: x[1]):
    print(f"  {name:22s}: RMSE = {rmse:.4f}")


── Итоговое сравнение RMSE (tuned) ──
  RandomForest          : RMSE = 1.1720
  GradientBoosting      : RMSE = 1.1958
  ElasticNet            : RMSE = 1.3176
  Ridge                 : RMSE = 1.3251


## Feature Importance

In [6]:
importances = gs_rf.best_estimator_.feature_importances_
top_idx = np.argsort(importances)[-20:]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh([X.columns[i] for i in top_idx], importances[top_idx], color="mediumseagreen")
ax.set_title("RandomForest: Топ-20 важных признаков (SI)")
ax.set_xlabel("Feature Importance")
plt.tight_layout()
plt.savefig("plots/si_feature_importance.png")
plt.close()
print("\nСохранён график: si_feature_importance.png")


Сохранён график: si_feature_importance.png


## Предсказание vs Факт

In [ ]:
y_pred = gs_gb.predict(X)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y, y_pred, alpha=0.3, s=10, color="mediumseagreen")
mn, mx = y.min(), y.max()
ax.plot([mn, mx], [mn, mx], "r--", lw=1.5)
ax.set_xlabel("log1p(SI) — факт")
ax.set_ylabel("log1p(SI) — предсказание")
ax.set_title(f"GradientBoosting (tuned): R²={r2_score(y, y_pred):.3f}")
plt.tight_layout()
plt.savefig("plots/si_pred_vs_actual.png")
plt.close()

## Итоговые выводы

- Целевая переменная SI имеет сильный перекос: среднее (72.5) значительно выше медианы (3.85), максимум достигает 15620, то есть распределение сильно вытянуто вправо.
- Линейные модели показывают низкое качество. Lasso и ElasticNet с дефолтными параметрами дают R² около нуля. Ridge немного лучше (R²=0.10).
- Настройка гиперпараметров улучшила линейные модели, но незначительно: лучший ElasticNet даёт RMSE=1.318.
- Лучший Random Forest достигает RMSE=1.172 и R²=0.33, GradientBoosting — RMSE=1.196.
- Разрыв между лучшей линейной (ElasticNet, RMSE=1.318) и лучшей древовидной (Random Forest, RMSE=1.172) составляет 0.146 по RMSE.
- Качество предсказания остаётся скромным: R² всего 0.33 даже у лучшей модели. Это говорит о том, что молекулярные дескрипторы объясняют лишь треть дисперсии логарифма селективности.
- Основная сложность — в сильном перекосе исходных данных: модель хорошо работает на типичных значениях, но вряд ли точно предскажет редкие выбросы с очень высокой селективностью.